# 관계 파악 차트 — 산점도 · 상관계수 · 히트맵


**직접 풀어보는 12문제:** 정답 코드는 넣지 않았습니다. 준비 코드만 제공하며 각 문제의 빈 셀에 풀이를 작성합니다. 점검 숫자는 아래 가상 데이터에 대한 값입니다. 이 노트북의 준비 데이터에만 점검 숫자를 적용하세요.

각 노트북은 독립적으로 사용할 수 있습니다. 위에서 아래로 진행하고, 실험에는 복사본을 사용하세요. 개별 문제에서 만든 결과는 다른 이름에 저장해 원본 변수를 덮어쓰지 않는 것을 권합니다.

## 처음 배우는 분을 위한 안내

산점도는 한 관측을 점 하나로 그립니다. X축·Y축에 비교할 숫자 열을 놓습니다. 상관계수는 직선 관계의 방향·강도를 −1~1로 요약하며 인과를 입증하지 않습니다. hue는 그룹별 색상, 히트맵은 값을 색으로 나타내는 표, pairplot은 여러 변수 쌍의 산점도 모음입니다.

**학습 순서:** 문제 1~4는 필수 기초, 여러 변수를 묶는 5~12는 선택 도전입니다. 문제 2의 online_clean은 이후 문제 6·7·10에서도 씁니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

## 실행 준비

Python 3와 `pandas`, `numpy`, `matplotlib`, `seaborn`이 필요합니다(확장 시각화 노트북은 `wordcloud`, `folium` 추가 필요). 없는 경우 별도 코드 셀에서 `%pip install pandas numpy matplotlib seaborn wordcloud folium`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 자동 생성됩니다.

Colab 또는 Jupyter에서 `.ipynb`를 열고 **준비 셀부터 순서대로** 실행하세요. 아래 코드는 한글 폰트 설정과 실습 데이터 생성만 수행합니다.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 고객별 주문 요약 가상 데이터 (10명)
summary = pd.DataFrame({
    'customer_id': [f'C{i:02d}' for i in range(1, 11)],
    'total_orders': [1, 2, 2, 3, 4, 5, 6, 7, 8, 10],
    'total_spent': [50, 90, 110, 140, 210, 240, 310, 350, 420, 520],
    'avg_order_value': [50.0, 45.0, 55.0, 46.67, 52.5, 48.0, 51.67, 50.0, 52.5, 52.0],
    'tier': ['Bronze', 'Bronze', 'Bronze', 'Silver', 'Silver',
             'Silver', 'Gold', 'Gold', 'Gold', 'Gold']
})

# 온라인 주문 가상 데이터 (결측치 1건 포함, 12행)
online = pd.DataFrame({
    'order_id': range(101, 113),
    'region': ['South', 'North', 'South', 'East', 'North', 'South',
               'East', 'North', 'South', 'East', 'North', 'South'],
    'quantity': [1, 2, 2, 3, 1, 4, 2, 3, 5, 2, 4, 3],
    'unit_price': [20.0, 30.0, 25.0, 40.0, None, 50.0, 35.0, 60.0, 45.0, 80.0, 70.0, 90.0],
    'discount': [0.00, 0.05, 0.00, 0.10, 0.05, 0.15, 0.05, 0.10, 0.20, 0.05, 0.15, 0.10],
    'delivery_days': [2, 3, 2, 5, 4, 6, 3, 7, 8, 4, 9, 5],
    'customer_rating': [4.8, 4.5, 4.7, 3.8, 4.2, 3.5, 4.4, 3.2, 3.0, 4.1, 2.8, 3.9],
    'revenue': [20.0, 57.0, 50.0, 108.0, None, 170.0, 66.5, 162.0, 180.0, 152.0, 238.0, 243.0]
})

# 연도·월별 주가/매출 피벗 히트맵용 데이터
monthly_perf = pd.DataFrame({
    'year': [2024, 2024, 2024, 2024, 2025, 2025, 2025, 2025],
    'month': [1, 2, 3, 4, 1, 2, 3, 4],
    'close_avg': [110.0, 140.0, 135.0, 160.0, 210.0, 240.0, 255.0, 280.0]
})

## 문제 1. 주문 횟수와 총 구매금액 산점도 [필수 기초]

`summary`의 `total_orders`(X축)와 `total_spent`(Y축)로 산점도(`plt.scatter`)를 그리세요. X축 이름, Y축 이름, 그래프 제목을 지정하고 점들의 분포 방향을 해석하세요.

**점검 기준:** 10명의 고객 데이터가 왼쪽 아래(`(1, 50)`)에서 오른쪽 위(`(10, 520)`)로 거의 직선에 가깝게 배열되어 강한 양의 선형 관계를 보입니다.

In [ ]:
# 문제 1 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 2. 결측치를 제거한 상품 가격과 주문 매출 산점도 [필수 기초]

`online`에서 `unit_price`와 `revenue`의 결측치를 제거한 복사본 `online_clean`을 만들고, X축 `unit_price`, Y축 `revenue`로 산점도를 그리세요.

**점검 기준:** 결측 1행(`order_id=105`)이 제외되어 유효 행 수는 11행입니다. 단가가 높을수록 주문 매출도 전반적으로 커지는 우상향 패턴이 나타납니다.

In [ ]:
# 문제 2 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 3. 산점도 위에 데이터 포인트 라벨(`annotate`) 표시하기 [필수 기초]

`summary`의 `total_orders`와 `total_spent` 산점도를 그린 뒤, 반복문과 `plt.annotate()`를 사용해 모든 점(또는 주문 횟수 상위 3명) 옆에 `customer_id` 라벨을 표시하세요.

**점검 기준:** 각 점 옆에 `C01`~`C10` 식별자가 표시되어 가장 우상단 극단 위치의 고객이 `C10(주문 10회, 520)`임을 그래프에서 바로 식별할 수 있습니다.

In [ ]:
# 문제 3 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 4. 주문 횟수와 총 구매금액의 피어슨 상관계수 계산 [필수 기초]

`summary['total_orders']`와 `summary['total_spent']`의 피어슨 상관계수를 `.corr()`로 계산해 소수점 넷째 자리까지 출력하고, 관계의 방향과 강도를 적으세요.

**점검 기준:** 상관계수는 약 `0.9982`로 `+1`에 매우 가까우며, 주문 횟수와 총 구매금액 사이에 매우 강한 양(+)의 선형 관계가 있습니다.

In [ ]:
# 문제 4 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 5. 상품 수량·단가와 매출의 상관계수 비교 [선택 도전]

`online`에서 결측치를 제거한 뒤 `quantity`와 `revenue`의 상관계수, `unit_price`와 `revenue`의 상관계수를 각각 구해 비교하세요.

**점검 기준:** 유효 11행 기준 `quantity`와 `revenue`의 상관계수는 약 `0.7460`, `unit_price`와 `revenue`의 상관계수는 약 `0.8750`으로 두 변수 모두 매출과 강한 양의 상관관계를 보입니다.

In [ ]:
# 문제 5 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 6. 범주별 색상(`hue`)을 구분한 산점도 (`sns.scatterplot`) [선택 도전]

`online_clean` 데이터에서 X축 `unit_price`, Y축 `revenue`, 색상 구분 `hue='region'`으로 `sns.scatterplot`을 그리고 지역별 분포를 확인하세요.

**점검 기준:** `South`, `North`, `East` 세 지역이 서로 다른 색상 점으로 구분되며 범례가 자동으로 생성됩니다.

In [ ]:
# 문제 6 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 7. 산점도와 단변량 분포를 결합한 `sns.jointplot` [선택 도전]

`online_clean` 데이터에서 `x='unit_price'`, `y='revenue'`, `hue='region'`으로 `sns.jointplot`을 실행해 중앙 산점도와 상단·우측 밀도 곡선을 함께 확인하세요.

**점검 기준:** 중앙에는 두 변수의 우상향 산점도가, 위쪽과 오른쪽 여백에는 지역별 `unit_price`와 `revenue`의 KDE 분포 곡선이 동시에 표시됩니다.

In [ ]:
# 문제 7 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 8. 주문 데이터 6개 수치형 변수 상관관계 히트맵 [선택 도전]

`online`의 `['quantity', 'unit_price', 'discount', 'delivery_days', 'customer_rating', 'revenue']` 6개 열에 대해 상관계수 행렬을 구하고 `sns.heatmap(annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0)`으로 시각화하세요.

**점검 기준:** `revenue`와 강한 양의 상관을 보이는 변수는 `unit_price(0.88)`, `delivery_days(0.83)`, `discount(0.77)`, `quantity(0.75)`이며, `customer_rating`과는 강한 음의 상관(`-0.82`)을 보입니다.

In [ ]:
# 문제 8 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 9. 고객 요약 지표 상관관계 히트맵과 대각선 값의 의미 [선택 도전]

`summary`의 `['total_orders', 'total_spent', 'avg_order_value']` 세 열의 상관계수 히트맵을 그리고, 대각선 값이 모두 `1.00`인 이유와 `total_orders` vs `avg_order_value` 상관계수가 낮은 이유를 설명하세요.

**점검 기준:** 대각선은 자기 자신과의 상관이므로 이 자료의 비상수 열에서는 `1.00`입니다. `total_orders`와 `total_spent`는 `1.00(0.998)`이지만, 1회당 평균 결제액(`avg_order_value`)은 주문 횟수와 무관하게 45~55 수준으로 일정해 상관계수가 약 `0.30`으로 낮게 나타납니다.

**주의:** 상수 열이나 유효 관측이 부족한 열의 상관계수는 NaN일 수 있습니다. 평균 결제액과 주문 횟수의 관계는 이 작은 자료에서 계산된 결과로만 해석합니다.

In [ ]:
# 문제 9 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 10. 다변량 관계 탐색을 위한 `sns.pairplot` [선택 도전]

`online_clean`에서 `['quantity', 'unit_price', 'delivery_days', 'customer_rating', 'revenue']` 열만 선택해 `sns.pairplot`을 그리고, 실무에서 전체 열을 무조건 넣지 않고 일부 열만 선택하는 이유를 적으세요.

**점검 기준:** 5개 변수의 `5×5` 산점도·히스토그램 격자가 생성됩니다. 변수 수가 `N`개일 때 차트가 `N²`개로 늘어나므로 핵심 수치형 변수만 추려서 실행해야 가독성과 속도를 확보할 수 있습니다.

In [ ]:
# 문제 10 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 11. 피벗 테이블(`pivot_table`) 기반 범주×범주 집계 히트맵 [선택 도전]

`monthly_perf` 데이터를 `index='year'`, `columns='month'`, `values='close_avg'`, `aggfunc='mean'`으로 피벗한 뒤 `sns.heatmap(annot=True, fmt='.1f', cmap='YlOrRd')`으로 시각화하세요.

**점검 기준:** `2행(2024·2025년) × 4열(1~4월)` 피벗 표가 생성되며, 2024년 1월(`110.0`)에서 2025년 4월(`280.0`)로 갈수록 색상이 진해지며 상승 추세가 한눈에 드러납니다.

In [ ]:
# 문제 11 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 12. 상관관계와 인과관계 비판적 해석하기 [선택 도전]

문제 5와 문제 8에서 `unit_price`와 `revenue`의 상관계수는 약 `0.88`, `delivery_days`와 `revenue`의 상관계수는 약 `0.83`으로 나타났습니다. 이 결과만 보고 ① '상품 가격을 올리면 매출이 무조건 증가한다', ② '배송을 늦게 할수록 매출이 오른다'고 주장할 수 없는 이유를 교란 변수 관점에서 서술하세요.

**점검 기준:** 매출(`revenue = quantity × unit_price × (1 - discount)`)에는 구매 수량과 할인율, 고가 상품군의 특성(대형 가전 등 고가·다량 주문이 배송도 오래 걸리는 현상 등)이라는 제3의 요인이 작용하므로 단순 상관을 인과로 해석하면 잘못된 의사결정을 내리게 됩니다.

In [ ]:
# 문제 12 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 선택 도전: 네 점의 선형 관계 확인

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_pairs = pd.DataFrame({'hours':[1,2,3,4],'score':[2,4,6,8]})
```

### 진행 순서

1. hours를 X축, score를 Y축으로 산점도를 그립니다.
2. 두 열의 corr()를 출력합니다.
3. 이 가상 규칙을 실제 공부시간의 인과 효과라고 주장할 수 없는 이유를 한 문장으로 적습니다.

**완료 기준:** 점 4개가 오른쪽 위로 향하는 직선 위에 있고 상관계수 1.0. 실제 현상을 검증한 자료는 아님.

## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 그래프에 제목(`title`), 축 이름(`xlabel`, `ylabel`), 다중 계열인 경우 범례(`legend`)를 표시했나요?
- 모든 점검 기준에 대해 수치 근거와 시각적 특징을 함께 남겼나요?
- 결측값 처리 여부와 표본 크기(또는 구간/윈도우 설정)를 확인했나요?
- 원본 데이터프레임 변수를 덮어쓰지 않고 안전하게 실습했나요?